In [1]:
import os
from dotenv import load_dotenv

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

C:\Users\thubz\AppData\Local\Temp\ipykernel_13768\330047604.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\thubz\AI-projects\AI-application\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OpenAi_API_key")


In [3]:

print("\nLoading documents...\n")

documents = []

data_folder = "data"

for file_name in os.listdir(data_folder):

    file_path = os.path.join(data_folder, file_name)

    loader = TextLoader(file_path, encoding="utf-8")

    docs = loader.load()

    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")


Loading documents...

Total documents loaded: 3


In [4]:
#chunk documents




splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap=50,
    separators=["\n\n","\n",","]
)


chunks = splitter.split_documents(documents)

In [20]:
#create embeddings

embedding_model = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-3-small"
)

In [6]:
#chroma vector database


vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="company_guidelines",
    persist_directory="chroma_db"
)


In [10]:
#retriver
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [12]:


user_input = """
hey bro,
i used chatgpt and github for this project
"""


print("\nUser Input:\n")
print(user_input)


User Input:


hey bro,
i used chatgpt and github for this project



In [14]:
#retrieve relevant documents
print("\nRetrieving relevant chunks...\n")

retrieved_docs = retriever.invoke(user_input)

for i, doc in enumerate(retrieved_docs, start=1):

    print(f"\n--- Chunk {i} ---\n")

    print(doc.page_content)


Retrieving relevant chunks...


--- Chunk 1 ---

14. PERSONAL DATA

Only request personal information when necessary.

Never ask customers to provide:
- Passwords
- Authentication codes
- Private security credentials
- Unnecessary sensitive information

15. INTERNAL COMMUNICATION

--- Chunk 2 ---

17. FINAL CHECK

Before sending a response, verify:
- Did I answer the question?
- Is the information supported?
- Did I avoid inventing details?
- Is the response easy to understand?
- Did I provide a useful next step when needed?

--- Chunk 3 ---

Avoid:
- Excessive jargon
- Complex sentence structures
- Excessive capitalization
- Excessive exclamation marks
- Decorative language
- Repetition

11. PUNCTUATION

Use normal punctuation.

Avoid excessive use of:
!!!
???
...


In [15]:
#combine retrieved context
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [16]:
prompt = ChatPromptTemplate.from_template(
    """
You are a professional company writing assistant.

Use the provided company guidelines to fix the user text.

Guidelines:
{context}

User Text:
{input}

Return only the corrected professional version.
"""
)


In [17]:
#create chain, llm
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4.1-mini",
    temperature=0
)


# =========================================================
# CREATE FINAL CHAIN
# =========================================================

chain = prompt | llm

In [18]:
print("\nGenerating final response...\n")

response = chain.invoke({
    "context": context,
    "input": user_input
})


Generating final response...



In [19]:

print("\n========== FINAL OUTPUT ==========\n")

print(response.content)


========== FINAL OUTPUT ==========

Hello,  
I used ChatGPT and GitHub for this project.
